In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from typing import Tuple

In [ ]:
from mcap.reader import make_reader

mcap_file_path_outer = "../data/Maps/outer.mcap"
mcap_file_path_inner_1 = "../data/Maps/inner1.mcap"
mcap_file_path_inner_2 = "../data/Maps/inner2.mcap"


with open(mcap_file_path_outer, "rb") as f:
    reader = make_reader(f)

    # 1. Zusammenfassung der Datei laden
    summary = reader.get_summary()

    if summary:
        print("--- ALLGEMEINE STATISTIKEN ---")
        stats = summary.statistics
        print(f"Nachrichten insgesamt: {stats.message_count}")
        print(f"Anzahl Kanäle/Topics: {stats.channel_count}")
        print(f"Startzeit (Nanosekunden): {stats.message_start_time}")
        print(f"Endzeit (Nanosekunden): {stats.message_end_time}")

        print("\n--- REGISTRIERTE SCHEMATA (Datentypen) ---")
        for schema_id, schema in summary.schemas.items():
            print(f"ID {schema_id}: {schema.name} (Typ: {schema.encoding})")

        print("\n--- AKTIVE KANÄLE (Topics) ---")
        for channel_id, channel in summary.channels.items():
            # Zugehöriges Schema finden
            schema_name = summary.schemas[channel.schema_id].name if channel.schema_id in summary.schemas else "Kein Schema"
            print(f"Topic: '{channel.topic}' | Typ: {schema_name} | Encoding: {channel.message_encoding}")
    else:
        print("Die Datei enthält keine Summary-Sektion. Sie muss sequenziell gescannt werden.")


In [ ]:
from mcap_ros2.decoder import DecoderFactory

def decode_msg(path: str, get_data: bool) -> Tuple[NDArray, NDArray] | Tuple[None, None]:
    x = []
    y = []

    with open(path, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])

        for i, (schema, channel, message, msg) in enumerate(
            reader.iter_decoded_messages(topics=["/mapping_tool/odometry", "/tf"])
        ):
            # print(channel.topic == "/tf")
            if channel.topic == "/tf":
                # position = msg.transforms[-1].transform.translation
                # print(type(position))
                # print(msg.transforms[-1])
                # print(position)
                # print("In here...\n")
                # print(msg)

                pass
            else:
                position = msg.pose.pose.position
                orientation = msg.pose.pose.orientation

            if get_data:
                x.append(position.x)
                y.append(position.y)
            else:
                # print(f"\nMessage {i}")
                # print(f"Reference frame: {msg.header.frame_id}")
                # print(f"Tracked frame:   {msg.child_frame_id}")
                # print(f"Stamp: {msg.header.stamp.sec} s + "
                #       f"{msg.header.stamp.nanosec} ns")
                # print(f"Position: x={position.x}, y={position.y}, z={position.z}")
                # print(f"Orientation: x={orientation.x}, y={orientation.y}, "
                #       f"z={orientation.z}, w={orientation.w}")

                if i >= 5:
                    break

    if get_data:
        return np.array(x), np.array(y)
    else:
        return None, None

decode_msg(mcap_file_path_outer, False)

In [ ]:
x_outer, y_outer = decode_msg(mcap_file_path_outer, True)

x_inner_1, y_inner_1 = decode_msg(mcap_file_path_inner_1, True)

x_inner_2, y_inner_2 = decode_msg(mcap_file_path_inner_2, True)

In [ ]:
plt.scatter(x_outer, y_outer, s=1)
plt.scatter(x_inner_1, y_inner_1, s=1)
plt.scatter(x_inner_2, y_inner_2, s=1)

plt.gca().set_aspect("equal", adjustable="box")

plt.xlabel("x")
plt.ylabel("y")

# plt.xlim(-4, 6)
# plt.ylim(-4, 6)

In [ ]:
# Loop over the path segments and average n consecutive points (close in time measurements) to obtain fewer
# points that are still a sequence and that have smaller positional errors (CLT)

print(len(x_outer))

def point_averages(x_array: NDArray, y_array: NDArray, n: int) -> Tuple[NDArray, NDArray]:
    x_averages = []
    y_averages = []

    max_index = int(len(x_array) / n)

    for i in range(max_index):
        x_average = np.mean(x_array[i * n: (i + 1) * n])
        y_average = np.mean(y_array[i * n: (i + 1) * n])

        x_averages.append(x_average)
        y_averages.append(y_average)

    return np.array(x_averages), np.array(y_averages)

In [ ]:
n_avg = 50
x_outer_avg, y_outer_avg = point_averages(x_outer, y_outer, n_avg)
x_inner_1_avg, y_inner_1_avg = point_averages(x_inner_1, y_inner_1, n_avg)
x_inner_2_avg, y_inner_2_avg = point_averages(x_inner_2, y_inner_2, n_avg)

In [ ]:
plt.scatter(x_outer_avg, y_outer_avg, s=1)
plt.scatter(x_inner_1_avg, y_inner_1_avg, s=1)
plt.scatter(x_inner_2_avg, y_inner_2_avg, s=1)

plt.gca().set_aspect("equal", adjustable="box")

# plt.ylim(3, 5)

plt.xlabel("x")
plt.ylabel("y")

In [ ]:
# Compute min and max values of the inner paths to get bounds and crossing regions

min_x_inner_1 = np.min(x_inner_1_avg)
max_x_inner_1 = np.max(x_inner_1_avg)
min_y_inner_1 = np.min(y_inner_1_avg)
max_y_inner_1 = np.max(y_inner_1_avg)

min_x_inner_2 = np.min(x_inner_2_avg)
max_x_inner_2 = np.max(x_inner_2_avg)
min_y_inner_2 = np.min(y_inner_2_avg)
max_y_inner_2 = np.max(y_inner_2_avg)


# Now compute max of outer to get outer / upper bounds on the map

min_x_outer = np.min(x_outer_avg)
max_x_outer = np.max(x_outer_avg)
min_y_outer = np.min(y_outer_avg)
max_y_outer = np.max(y_outer_avg)

print(max_x_outer, min_x_outer)

In [ ]:
# Top outer
plt.hlines(max_y_outer, min_x_outer, max_x_outer, ls="--", color='black', alpha=0.5)
# Bottom outer
plt.hlines(min_y_outer, min_x_outer, max_x_outer, ls="--", color='black', alpha=0.5)
# Left outer
plt.vlines(min_x_outer, min_y_outer, max_y_outer, ls="--", color='black', alpha=0.5)
# Right outer
plt.vlines(max_x_outer, min_y_outer, max_y_outer, ls="--", color='black', alpha=0.5)

# Left middle
plt.vlines(min_x_inner_1, max_y_inner_2, min_y_inner_1, ls="--", color='black', alpha=0.5)
# Right middle
plt.vlines(max_x_inner_1, max_y_inner_2, min_y_inner_1, ls="--", color='black', alpha=0.5)

# Left box top
plt.hlines(min_y_inner_1, min_x_outer, min_x_inner_1, ls="--", color='black', alpha=0.5)
# Left box bottom
plt.hlines(max_y_inner_2, min_x_outer, min_x_inner_1, ls="--", color='black', alpha=0.5)
# Right box top
plt.hlines(min_y_inner_1, max_x_inner_2, max_x_outer, ls="--", color='black', alpha=0.5)
# Right box bottom
plt.hlines(max_y_inner_2, max_x_inner_2, max_x_outer, ls="--", color='black', alpha=0.5)


# Averaged measured points
plt.scatter(x_outer_avg, y_outer_avg, s=1)
plt.scatter(x_inner_1_avg, y_inner_1_avg, s=1)
plt.scatter(x_inner_2_avg, y_inner_2_avg, s=1)


plt.gca().set_aspect("equal", adjustable="box")

# plt.ylim(-3.5, -2.5)

plt.xlabel("x")
plt.ylabel("y")

In [ ]:
# Interesting approach to compute the middle lines. Only possible due to inherent symmetry of the track:
# Take the inner ones and make all points extend outwards until they are half the way to the outer / other
# inner. This would probably work but still maybe be not optimal (additional post-processing required).

# Compute the center points of inner_1 and inner_2
center_x_inner_1 = (min_x_inner_1 + max_x_inner_1) / 2
center_y_inner_1 = (min_y_inner_1 + max_y_inner_1) / 2

# Now, we want to loop over all points, and move them in the direction from the center points by the width
# of one track (so half of the distance between the inner and the outer paths (at most places)).

# 1. Compute the width of one track (i.e. the 'magnitude of the addition') — this could be averaged, probably
#    even over 4 values since we have two boxes (that should both be squares).
track_width = (max_x_outer - max_x_inner_1) / 2

# --> This kind of won't work without tangents and then there is probably actually a simpler way...

In [ ]:
# Now, we try to set the point in the center of the map and then loop over the outer array and will here
# compute (or approximate) the tangent with n (initially maybe n = 1) points to both sides of the current
# point. And then, we can just compute the track separation line point by 'moving it by one track with
# inward (perpendicular to the tangent). And actually, we'll likely don't even need a center point for this.

# Function that takes n points and computes the tangent (i.e. a linear regression)


# And quick try:
# pts = np.column_stack((x_outer_avg, y_outer_avg))
# raw_tangents = 0.5 * (np.roll(pts, -1, axis=0) - np.roll(pts, 1, axis=0))
# norms = np.linalg.norm(raw_tangents, axis=1, keepdims=True)
# raw_tangents / np.where(norms == 0, 1.0, norms)
#
# middle = pts - 0
#
# print(pts.shape)
# print(raw_tangents)

In [ ]:
def three_point_tangents_parametric(pts):
    """
    Method A: Parametric 3-point linear regression over t = [-1, 0, 1].
    Algebraically equivalent to (P_{i+1} - P_{i-1}) / 2.
    Returns unit tangent vectors of shape (N, 2).
    """
    prev_pts = np.roll(pts, 1, axis=0)   # P_{i-1} (wraps around automatically)
    next_pts = np.roll(pts, -1, axis=0)  # P_{i+1}

    raw_tangents = 0.5 * (next_pts - prev_pts)
    norms = np.linalg.norm(raw_tangents, axis=1, keepdims=True)
    return raw_tangents / np.where(norms == 0, 1.0, norms)

In [ ]:
points = np.column_stack((x_outer_avg, y_outer_avg))

tangents = three_point_tangents_parametric(points)

# print(tangents)

normals_ccw = np.column_stack([-tangents[:, 1], tangents[:, 0]])

print(points.shape, tangents.shape)

track_boundary = points + track_width * normals_ccw

# Remove first and last due to tangent direction issues
track_boundary = track_boundary[1:-1]

In [ ]:
# Clean the track boundary array using masking such that there are no lines in the two boxes
outer_track_boundary_mask = (track_boundary[:, 1] < max_y_inner_2) | (track_boundary[:, 1] > min_y_inner_1)

print(min_y_inner_1, max_y_inner_2)


track_boundary = track_boundary[outer_track_boundary_mask]

In [ ]:
# Add the middle line
middle_x = np.linspace(min_x_inner_1, max_x_inner_1, num=int((max_x_inner_1 - min_x_inner_1) * 10))
middle_y = np.array([(min_y_inner_1 + max_y_inner_2) / 2 for _ in middle_x])

In [ ]:
# Bounds
# Top outer
plt.hlines(max_y_outer, min_x_outer, max_x_outer, ls="--", color='black', alpha=0.5)
# Bottom outer
plt.hlines(min_y_outer, min_x_outer, max_x_outer, ls="--", color='black', alpha=0.5)
# Left outer
plt.vlines(min_x_outer, min_y_outer, max_y_outer, ls="--", color='black', alpha=0.5)
# Right outer
plt.vlines(max_x_outer, min_y_outer, max_y_outer, ls="--", color='black', alpha=0.5)

# Left middle
plt.vlines(min_x_inner_1, max_y_inner_2, min_y_inner_1, ls="--", color='black', alpha=0.5)
# Right middle
plt.vlines(max_x_inner_1, max_y_inner_2, min_y_inner_1, ls="--", color='black', alpha=0.5)

# Left box top
plt.hlines(min_y_inner_1, min_x_outer, min_x_inner_1, ls="--", color='black', alpha=0.5)
# Left box bottom
plt.hlines(max_y_inner_2, min_x_outer, min_x_inner_1, ls="--", color='black', alpha=0.5)
# Right box top
plt.hlines(min_y_inner_1, max_x_inner_2, max_x_outer, ls="--", color='black', alpha=0.5)
# Right box bottom
plt.hlines(max_y_inner_2, max_x_inner_2, max_x_outer, ls="--", color='black', alpha=0.5)

# Averaged measured points
plt.scatter(x_outer_avg, y_outer_avg, s=1)
plt.scatter(x_inner_1_avg, y_inner_1_avg, s=1)
plt.scatter(x_inner_2_avg, y_inner_2_avg, s=1)

plt.scatter(track_boundary[1:-1, 0], track_boundary[1:-1, 1], s=1, c='r')
plt.scatter(middle_x, middle_y, s=1, c='r')

plt.gca().set_aspect("equal", adjustable="box")

# plt.ylim(-3.5, -2.5)

plt.xlabel("x")
plt.ylabel("y")

In [ ]:
# Adjust x so all values are positive to avoid 'wrappings' in the grid map. Probably just need to invert
# the transformation at the end. Apply it to all to ensure consistency.
y_outer += 3.5
y_inner_1 += 3.5
y_inner_2 += 3.5

In [ ]:
# Try to rasterize the map such that we have a boolean map of 'line' / 'no line'. This way,
# we can then easier create the points to plot by getting rid duplicates and similars.
# First however count occurrences.

y_range = [0, 7.5]
x_range = [0.5, 6]

resolution = 0.1

x_dim = int(round(x_range[1] - x_range[0]) / resolution)
y_dim = int(round(y_range[1] - y_range[0]) / resolution)

grid_map = np.zeros((x_dim, y_dim))

print(np.shape(grid_map))

In [ ]:
# Now we essentially want to loop over the points and increment the grid point where the value lies
# For this a small helper function might be useful.

# Example: x = 1.35, resolution = 0.1. Then x / resolution = 13.5 and since x < resolution should be at index 0,
# this means that x_index = int(x / resolution), i.e. rounding down / truncating.

def get_grid_position(x: float, y: float) -> Tuple[int, int]:
    # The -1 kind of makes no sense but just trying it out.
    x_index = int(x / resolution) - 1
    y_index = int(y / resolution) - 1

    return x_index, y_index

# Then now we want a function that loops over the array and updates the entries accordingly
def update_grid_map(x_array: NDArray, y_array: NDArray) -> None:
    for x, y in zip(x_array, y_array):
        x_index, y_index = get_grid_position(x, y)
        grid_map[x_index, y_index] += 1

    return None

In [ ]:
update_grid_map(x_outer, y_outer)

In [ ]:
print(grid_map)

In [ ]:
plt.imshow(grid_map, cmap="viridis")
plt.colorbar()
# plt.axis("off")  # Drop this line if you need row/column indices to locate anomalies

# plt.gca().set_aspect("equal", adjustable="box")

plt.show()

In [ ]:
# Now we want to extract the new points from the grid map again. For this we use the center (can later be
# improved by shifted overlays / averaging).

# Actually, it seems like it is not that easy to extract a sequence of points. Only again a set (but unordered).
# However, with a specified starting point and some proximity heuristics, this might just work... The initial
# attempt will only return a set for now.
def extract_from_map(map_array: NDArray, bound: int = 1) -> NDArray:
    """Extract entries larger than zero (or the specified min bound (inclusive)) from the grid map and turn
    them back to 2d points. Returns an array of x,y coordinates."""

    shape = np.shape(map_array)
    point_set = []

    for i in range(shape[0]):
        for j in range(shape[1]):
            if map_array[i, j] >= bound:
                x = resolution / 2 + i * resolution
                y = resolution / 2 + j * resolution
                point_set.append((x, y))

    return np.array(point_set)

In [ ]:
points = extract_from_map(grid_map)

In [ ]:
plt.scatter(points[:, 0], points[:, 1], s=1, marker='s')
plt.gca().set_aspect("equal", adjustable="box")

In [ ]:
# Then to maybe average everything a bit, we could create the same map with small shifts in x and y try to build
# a more accurate representation from this.